![Universidad Espíritu Santo](https://raw.githubusercontent.com/andresrubiop/miar0525-estudiantes/main/utils/logo-uees-color.png)

<div style="background:#821436;color:#FFFFFF;padding:14px 18px;border-radius:10px;margin:6px 0 12px 0"><div style="font-size:12px;letter-spacing:.08em;text-transform:uppercase;opacity:.9">Aprendizaje Automático · MIAR0525 · Semana 3 · Tarea 3 · 20 % · entrega hasta el vie 16/10, 23:59</div><div style="font-size:22px;font-weight:700;margin-top:4px">T3 · Reducción de dimensionalidad, anomalías e impacto por subgrupos</div><div style="font-size:12px;opacity:.9;margin-top:4px">Postgrado · Maestría en Inteligencia Artificial · UEES</div></div>

| | |
|---|---|
| **Qué entregas** | **Este mismo notebook**, ejecutado de principio a fin y guardado con sus salidas, renombrado `T3_Apellido_Nombre.ipynb`. No hay informe en PDF ni presentación aparte: el resumen ejecutivo de la sección 7 hace las veces de presentación. |
| **Caso** | Un emisor de tarjetas de crédito quiere entender los perfiles de pago de sus clientes y detectar comportamientos inusuales. |
| **Datos** | default-of-credit-card-clients (OpenML `data_id=42477`): 30 000 clientes de Taiwán, 23 variables (`x1`–`x23`) y `y` = incumplimiento el mes siguiente (22.1 %). |
| **Variables sensibles** | Sexo, educación, estado civil y edad: **no entran** a la segmentación ni a los detectores; se conservan para el análisis por subgrupos. |
| **Resultado de aprendizaje** | RDA2 · competencias CG-G1 y CE-G2 |
| **Puedes reutilizar** | E3.1 (K-means y validación) · E3.2 (DBSCAN) · E3.3 (PCA y t-SNE) · E3.4 (anomalías) |

## Cómo se califica

Cada sección es un criterio de la rúbrica y lleva su puntaje en el título. En cada una hay **Qué hacer**, celdas de
código con `# TODO` y una celda **Tu análisis** que debes responder: *el código que corre sin análisis no suma puntos.*
En esta tarea, el análisis por subgrupos vale **25 de los 100 puntos**.

**Uso de IA.** Permitida (agéntica o de chat) **si la declaras en la sección 8**, indicando en qué secciones la usaste.

**Antes de entregar**: *Kernel → Restart & Run All*, revisa que todas las celdas tengan salida y guarda con tu nombre.

## 0 · Configuración, datos y diccionario de variables (obligatorio)

En OpenML las columnas se llaman `x1` a `x23`. Esta celda las renombra según el conjunto original
(Yeh y Lien, 2009): montos en dólares taiwaneses y seis meses de historia, de abril a septiembre de 2005.

In [ ]:
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from cycler import cycler
from sklearn.datasets import fetch_openml

SEED = 2026
UEES = {"vino": "#821436", "azul": "#1F6F8B", "ocre": "#C28E0E", "verde": "#3A7D44", "gris": "#77787B"}
plt.rcParams.update({
    "axes.prop_cycle": cycler(color=list(UEES.values())), "axes.titlecolor": UEES["vino"],
    "axes.titleweight": "bold", "axes.edgecolor": UEES["gris"], "axes.grid": True, "grid.color": "#EEE8EA",
    "axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 110, "legend.frameon": False,
})
print(f"Python {sys.version.split()[0]} · scikit-learn {sklearn.__version__} · semilla {SEED}")

tc = fetch_openml(data_id=42477, as_frame=True).frame
meses = ["sep", "ago", "jul", "jun", "may", "abr"]
nombres = {"x1": "cupo", "x2": "sexo", "x3": "educacion", "x4": "estado_civil", "x5": "edad"}
nombres |= {f"x{6 + i}": f"estado_pago_{m}" for i, m in enumerate(meses)}
nombres |= {f"x{12 + i}": f"factura_{m}" for i, m in enumerate(meses)}
nombres |= {f"x{18 + i}": f"pago_{m}" for i, m in enumerate(meses)}
tc = tc.rename(columns=nombres).rename(columns={"y": "incumple"})
tc["incumple"] = tc["incumple"].astype(int)

SENSIBLES = ["sexo", "educacion", "estado_civil", "edad"]
print(f"{len(tc)} clientes · {tc.shape[1] - 1} variables · incumplimiento {tc['incumple'].mean():.1%}")
tc.head(3)

## 1 · Preparación y escalamiento justificado · 10 puntos

**Qué hacer:**

1. Revisa los códigos no documentados (`educacion` 0, 5 y 6; `estado_civil` 0; `estado_pago` −2 y 0) y **decide y
   documenta** qué haces con ellos (agruparlos en "otros" es razonable).
2. Arma la matriz de segmentación **sin** las variables sensibles ni `incumple`.
3. Elige y justifica el escalamiento (estándar, robusto o por cuantiles): los montos tienen colas muy largas.

*Reutiliza:* el escalamiento de E3.1 · Nivel 3.

In [ ]:
from sklearn.preprocessing import QuantileTransformer, RobustScaler, StandardScaler

# TODO: tratamiento de los códigos no documentados y matriz de segmentación (sin sensibles ni incumple).
# Xs = ...

# TODO: escalamiento justificado.
# Z = ...

In [ ]:
# Autoverificación de la sección 1
assert not set(SENSIBLES) & set(Xs.columns), "Las variables sensibles no deben entrar a la segmentación."
assert "incumple" not in Xs.columns, "La etiqueta no se usa para agrupar."
print("✓ matriz de segmentación:", Xs.shape, "· variables:", list(Xs.columns)[:6], "…")

**Tu análisis (sección 1).** ¿Qué hiciste con los códigos no documentados y por qué? ¿Qué escalamiento elegiste y
qué habría pasado con otro?

*(escribe aquí)*

## 2 · PCA: varianza, reconstrucción y uso · 15 puntos

**Qué hacer:**

1. Ajusta PCA sobre los datos escalados y grafica la **varianza explicada acumulada**.
2. Elige el número de componentes con un criterio explícito (por ejemplo, 90 % de varianza) y reporta el **error de
   reconstrucción**.
3. Explica qué representan las dos primeras componentes mirando sus cargas (`components_`).

*Reutiliza:* E3.3 · Niveles 1 a 3.

In [ ]:
from sklearn.decomposition import PCA

# TODO: PCA, varianza explicada acumulada, número de componentes elegido y error de reconstrucción.

**Tu análisis (sección 2).** ¿Cuántas componentes necesitas y por qué? ¿Qué mide la primera componente?

*(escribe aquí)*

## 3 · t-SNE, solo exploratorio · 10 puntos

**Qué hacer:**

1. Toma una muestra (2000–5000 clientes) y calcula t-SNE con **al menos dos perplejidades** (por ejemplo, 10 y 40),
   partiendo de las componentes de PCA.
2. Muestra los mapas lado a lado.
3. Escribe qué **no** se puede concluir de ellos.

*Reutiliza:* E3.3 · Nivel 4 y la animación A3.6.

In [ ]:
from sklearn.manifold import TSNE

# TODO: t-SNE con dos perplejidades sobre una muestra, a partir de PCA.

**Tu análisis (sección 3).** ¿Qué se repite entre las dos perplejidades y qué cambia? ¿Qué afirmaciones serían
incorrectas a partir de estos mapas (tamaños, distancias, densidad)?

*(escribe aquí)*

## 4 · Segmentación validada y estable · 15 puntos

**Qué hacer:**

1. Segmenta con **K-means o DBSCAN** (justifica la elección y los parámetros: codo, silhouette o k-distancia).
2. Valida: silhouette o Davies-Bouldin, **estabilidad** (ARI entre al menos tres muestras o semillas) y
   **perfiles** de cada segmento (medianas de las variables principales).
3. Ponle a cada segmento un nombre descriptivo de una frase.

*Reutiliza:* E3.1 · Niveles 2 y 3 (validación y perfiles) o E3.2 (DBSCAN).

In [ ]:
from sklearn.cluster import DBSCAN, KMeans
from sklearn.metrics import adjusted_rand_score, davies_bouldin_score, silhouette_score

# TODO: segmentación, validación interna, estabilidad y perfiles con nombres.

**Tu análisis (sección 4).** ¿Cuántos segmentos y por qué? ¿Qué tan estable es la solución? Describe cada segmento
en una frase.

*(escribe aquí)*

## 5 · Anomalías: Isolation Forest frente a LOF · 15 puntos

**Qué hacer:**

1. Aplica **Isolation Forest** y **LOF** sobre los mismos datos escalados.
2. Compara cuánto coinciden (por ejemplo, con el índice de Jaccard entre los conjuntos marcados) y en qué se
   diferencian los clientes que marca cada uno.
3. Justifica el **umbral operativo**: cuántas alertas por cada 1000 clientes puede revisar el equipo.
4. Como validación externa, compara la tasa de incumplimiento entre los marcados y el resto.

*Reutiliza:* E3.4 · Niveles 2 a 4.

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor

# TODO: Isolation Forest y LOF, comparación, umbral por capacidad y validación externa con incumple.

**Tu análisis (sección 5).** ¿En qué se parecen y en qué se diferencian los dos detectores? ¿Qué umbral elegiste y
con qué criterio? ¿Los clientes marcados incumplen más que el resto?

*(escribe aquí)*

## 6 · Composición por subgrupos y riesgos éticos · 25 puntos

**Qué hacer:**

1. Para **cada segmento** y para el conjunto de **clientes marcados como anómalos**, calcula la proporción de cada
   grupo (sexo, educación y edad en tramos) y la **razón de representación** = proporción en el grupo / proporción
   en la población.
2. Marca los casos fuera de la banda 0.8–1.25 y revisa si esos desbalances tienen sentido o son señal de un proxy.
3. Calcula la **tasa de marcado** de cada grupo en la detección de anomalías y su razón frente al grupo de referencia.
4. Discute: ¿qué proxies podrían estar actuando? ¿Qué usos de esta segmentación serían aceptables y cuáles no?
   ¿Qué recomendarías (revisión humana, no usarla para precios, etc.)?

*Reutiliza:* E3.1 · Nivel 4 (¿la segmentación discrimina?) y M3 §9.

In [ ]:
# TODO: tablas de composición y razón de representación por segmento y en las anomalías.

**Tu análisis (sección 6).** Responde las cuatro preguntas de arriba. Esta sección vale 25 puntos: sé concreto y
apóyate en tus tablas.

*(escribe aquí)*

## 7 · Resumen ejecutivo y limitaciones · 10 puntos

**Qué hacer:** escribe entre 200 y 300 palabras dirigidas al área de negocio, como si fuera la presentación que
harías en 8 diapositivas: qué segmentos encontraste, qué detecta el sistema de anomalías, qué desbalances por
subgrupo aparecieron, qué usos recomiendas y cuáles no, y las tres limitaciones principales del análisis.

**Tu resumen ejecutivo.**

*(escribe aquí)*

## 8 · Declaración de uso de IA (obligatoria)

Completa la tabla. Si no usaste IA, escribe "No usé IA" y firma igual (norma f del sílabo).

| | |
|---|---|
| **Herramientas** | *(por ejemplo: ChatGPT, Claude Code; o "ninguna")* |
| **Secciones donde la usé** | *(por ejemplo: sección 3 para los gráficos de t-SNE)* |
| **Para qué** | *(escribir código, depurar, redactar el análisis, revisar mi interpretación…)* |
| **Prompts relevantes** | *(pega los 2 o 3 más importantes)* |
| **Qué verifiqué yo** | *(ejecuté todo de cero, comprobé la estabilidad con otras semillas, revisé las tablas…)* |
| **Qué corregí o descarté** | *(qué propuso la IA que no usaste y por qué)* |

**Autoría.** El análisis, las decisiones y las conclusiones de este notebook son míos.

Nombre: *(tu nombre)* · Fecha: *(fecha de entrega)*

## Lista de cotejo antes de entregar

- [ ] Reinicié el kernel y ejecuté todo de arriba hacia abajo, sin errores.
- [ ] Las variables sensibles y `incumple` no entraron a la segmentación ni a los detectores.
- [ ] Mostré al menos dos perplejidades de t-SNE y advertí qué no se puede leer en esos mapas.
- [ ] Reporté estabilidad (ARI) y perfiles con nombre para cada segmento.
- [ ] Comparé Isolation Forest y LOF, con un umbral justificado por capacidad de revisión.
- [ ] La sección 6 tiene tablas por subgrupo con razones de representación y una discusión de riesgos.
- [ ] Respondí todas las celdas "Tu análisis" y completé la declaración de uso de IA.
- [ ] Guardé el archivo como `T3_Apellido_Nombre.ipynb` y lo subí al LMS.

In [ ]:
from datetime import datetime

print(f"Notebook ejecutado el {datetime.now():%Y-%m-%d %H:%M} · scikit-learn {sklearn.__version__} · semilla {SEED}")